# V4 — Why Subsea Internet Cables Are Wrapped in Steel (Blender + Eevee on Colab)

Blender short for @ExplainerMaan: scripted geometry (bpy), own-built 3D
ocean sets, one flat 2D bank clerk, rendered here with Eevee on a Colab T4 GPU.

**What this notebook does, in order:**
1. Mounts Drive + checks a GPU is attached (fails fast if not).
2. Installs Blender 4.2.23 + the system GL libs it needs.
3. Clones the project repo (`attempss7-tech/blender-cables-short`).
4. Renders **13 Eevee test frames (one per shot)** — **STOP and inspect these**
   before the full render (emission-driven actions — arc fade, shark-bite EM
   flare, screens dimming, city lights going out — were never seen locally).
5. Renders all 1,418 frames in **3 shards**, saving each batch to Drive
   immediately (a dropped session loses at most one batch, ~5 min).
6. Assembles the final mp4 (fades + sound mix) and verifies it.

**Before you run:** Runtime > Change runtime type > **T4 GPU** > Save.
Full click-path + what to do if a session drops: see `COLAB_RUN.md` in the repo.


In [ ]:
# Setup: mount Drive, check the GPU, define paths + a streaming run() helper.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, shutil, glob, json, time

REPO_DIR = '/content/blender-cables-short'
BLENDER = '/content/blender-4.2.23-linux-x64/blender'
DRIVE_ROOT = '/content/drive/MyDrive/cables_blender'
DRIVE_FRAMES = DRIVE_ROOT + '/frames'
DRIVE_TEST = DRIVE_ROOT + '/test_frames'
DRIVE_FINAL = DRIVE_ROOT + '/final'
LOCAL_FRAMES = '/content/frames'

def run(cmd, check=True):
    print('+', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed ({p.returncode}): {" ".join(cmd)}')
    return ''.join(out)

# GPU check — fail fast with a clear message if there is no GPU.
try:
    smi = run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
    print('GPU OK:', smi.strip())
except Exception:
    raise SystemExit('NO GPU ATTACHED — Runtime > Change runtime type > T4 GPU, then re-run this cell.')
for d in (DRIVE_FRAMES, DRIVE_TEST, DRIVE_FINAL, LOCAL_FRAMES):
    os.makedirs(d, exist_ok=True)
print('Paths ready.')


In [ ]:
# Install system GL libs + Blender 4.2.23 (same version the project was built with).
# Takes ~2-4 min. Re-running is safe (skips the download if already extracted).
run(['apt-get', 'update', '-qq'])
run(['env', 'DEBIAN_FRONTEND=noninteractive', 'apt-get', 'install', '-y', '-qq',
     'libegl1', 'libgl1', 'libegl-mesa0'])
if not os.path.exists(BLENDER):
    run(['curl', '-sSL', '-o', '/content/blender.tar.xz',
         'https://download.blender.org/release/Blender4.2/blender-4.2.23-linux-x64.tar.xz'])
    run(['tar', '-xf', '/content/blender.tar.xz', '-C', '/content'])
    os.remove('/content/blender.tar.xz')
run([BLENDER, '--version'])
print('Blender ready:', BLENDER)

# EGL FIX (required on Colab, proven 2026-11-12): Colab ships the NVIDIA EGL
# library at /usr/lib64-nvidia/libEGL_nvidia.so.0 but NO vendor file for it, so
# Blender's EGL context fails (EGL_BAD_MATCH) and Eevee silently falls back to
# CPU/llvmpipe at ~5 min/frame. Registering the vendor file makes Eevee use the
# GPU (~3.5 s/frame on a T4). The file lives only in this session's filesystem,
# so this cell must run again after any session drop (recovery re-runs setup).
import glob as _glob
_nvidia_egl = (_glob.glob('/usr/lib64-nvidia/libEGL_nvidia.so.0')
               + _glob.glob('/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0'))
if _nvidia_egl:
    os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
    with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as _f:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": "libEGL_nvidia.so.0"}}, _f, indent=2)
    print('EGL vendor file written (NVIDIA EGL at', _nvidia_egl[0] + ')')
else:
    print('WARNING: libEGL_nvidia.so.0 not found — Eevee may fall back to CPU.')
    print('Check the test-frames timings: minutes/frame means CPU fallback.')


In [ ]:
# Clone the project repo (or update it if already cloned).
if os.path.exists(REPO_DIR + '/.git'):
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    run(['git', 'clone', 'https://github.com/attempss7-tech/blender-cables-short.git', REPO_DIR])
run(['ls', '-la', REPO_DIR + '/scenes/', REPO_DIR + '/assets/'])


In [ ]:
# GPU probe + Eevee TEST FRAMES (one per shot — 13 frames).
# This is the mandatory look check: Eevee was never validated locally (no GPU
# there). Inspect the 13 frames below BEFORE running any shard cell.
run(['python3', f"{REPO_DIR}/scripts/colab_render.py",
     '--blender', BLENDER, '--gpu-probe'])
run(['python3', f"{REPO_DIR}/scripts/colab_render.py",
     '--blender', BLENDER, '--blend', f"{REPO_DIR}/scenes/cables.blend",
     '--out', '/content/test_frames', '--frames', '120,190,300,430,530,630,680,745,920,1000,1160,1290,1400'])
for f in glob.glob('/content/test_frames/*.png'):
    shutil.copy2(f, DRIVE_TEST + '/' + os.path.basename(f))
print('Test frames saved to Drive:', DRIVE_TEST)

from IPython.display import Image, display
for fr in [120, 190, 300, 430, 530, 630, 680, 745, 920, 1000, 1160, 1290, 1400]:
    p = f'/content/test_frames/f{fr:04d}.png'
    print(f'--- frame {fr} ---')
    display(Image(p, width=270))


## ⏸ STOP — inspect the 13 test frames above before continuing

Check each frame for the phrase's enacted action (one per shot):
- **f120** satellite + signal arc fading out (dim, almost gone) — "outer space"
- **f190** camera diving through the water column past fish toward the mud
- **f300** glowing cable lines stretched across the ocean floor map
- **f430** data pulses racing along glowing fibers inside the cable core
- **f530** human hair beside the glowing glass fiber, nearly the same thinness
- **f630** pressure arrows closing in, cable squeezed — "crushing pressure"
- **f680** seafloor plates shifted apart, crack open under the cable
- **f745** shark biting the cable, EM rings flaring around the bite
- **f920** jelly layer wrapped on, copper wrapping on — rings visible on the cable
- **f1000** steel armor wrapping on (tar comes after)
- **f1160** three cables snapped at once, ends whipped apart, sparks
- **f1290** bank screens + cloud icons freezing, frost on the screens, 2D clerk frozen mid-point
- **f1400** city going dark — most lights out, a few still on

**If the deep shots (f630–f745) look too DARK:** run the fix cell below (raises
world strength + sun energy), then re-run the test-frames cell.
**If all 13 frames look right:** skip the fix cell and run the 3 shard cells.


In [ ]:
# FIX CELL — ONLY run if the deep-sea test frames look too dark.
# Raises world strength and sun energy by 1.5x and re-saves the .blend,
# then re-renders ONE frame (f745, the shark bite) to confirm the fix.
import textwrap
fix = textwrap.dedent('''
    import bpy
    sc = bpy.context.scene
    sc.world.node_tree.nodes["Background"].inputs[1].default_value *= 1.5
    sun = bpy.data.objects.get("Sun")
    if sun:
        sun.data.energy *= 1.5
    print("world strength x1.5, sun energy x1.5")
    bpy.ops.wm.save_mainfile()
''')
open('/tmp/fix_brightness.py', 'w').write(fix)
run([BLENDER, '-b', f'{REPO_DIR}/scenes/cables.blend',
     '--python', '/tmp/fix_brightness.py'])
run(['python3', f'{REPO_DIR}/scripts/colab_render.py',
     '--blender', BLENDER, '--blend', f'{REPO_DIR}/scenes/cables.blend',
     '--out', '/content/test_frames', '--frames', '745'])
from IPython.display import Image, display
display(Image('/content/test_frames/f0745.png', width=270))
print('If f745 now reads clearly (shark + cable + EM rings visible), the fix worked.')
print('Re-run the test-frames cell to confirm all 13, then continue to the shards.')


In [ ]:
# SHARD 1: frames 0-472 (473 frames, 5 batches of 95).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 0, 472, 95
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/cables.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 1 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# SHARD 2: frames 473-945 (473 frames, 5 batches of 95).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 473, 945, 95
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/cables.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 2 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# SHARD 3: frames 946-1417 (472 frames, 5 batches of 95).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 946, 1417, 95
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/cables.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 3 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# ASSEMBLY: verify all 1,418 frames, then ffmpeg -> final mp4.
# Fades (ffmpeg, not baked into the scene): fade-in f0-14 (0.5s),
# fade-out f1403-1417 (0.5s from 46.756s, INSIDE the final voice — the last
# word ends at 47.04s, so the fade never runs over dead silence).
# Mix is already -14.5 LUFS — mux as-is.
missing = [f for f in range(1418) if not os.path.exists(f'{DRIVE_FRAMES}/f{f:04d}.png')]
print(f'Frames on Drive: {1418 - len(missing)}/1418, missing: {missing[:20]}')
assert not missing, f'{len(missing)} frames missing — re-run the shard cell(s) covering them.'
run(['ffmpeg', '-y',
     '-framerate', '30', '-i', f'{DRIVE_FRAMES}/f%04d.png',
     '-i', f'{REPO_DIR}/assets/mix.wav',
     '-vf', 'fade=t=in:st=0:d=0.5,fade=t=out:st=46.756:d=0.5,format=yuv420p',
     '-c:v', 'libx264', '-crf', '17', '-preset', 'medium',
     '-c:a', 'aac', '-b:a', '192k', '-shortest', '-movflags', '+faststart',
     '/content/cables_short.mp4'])
shutil.copy2('/content/cables_short.mp4', DRIVE_FINAL + '/cables_short.mp4')
print('Final video:', DRIVE_FINAL + '/cables_short.mp4')


In [ ]:
# VERIFICATION: ffprobe + loudness of the final mp4.
run(['ffprobe', '-v', 'error', '-show_entries',
     'format=duration,size:stream=width,height,r_frame_rate,codec_name',
     '-of', 'default=noprint_wrappers=1', DRIVE_FINAL + '/cables_short.mp4'])
run(['ffmpeg', '-i', DRIVE_FINAL + '/cables_short.mp4',
     '-vn', '-af', 'ebur128', '-f', 'null', '-'])
print('Expected: ~47.26s, 1080x1920, 30fps, h264+AAC, ~-14.5 LUFS.')


## Done

The final video is at `MyDrive/cables_blender/final/cables_short.mp4`,
with all 1,418 frames kept at `MyDrive/cables_blender/frames/` and the Eevee
test frames at `MyDrive/cables_blender/test_frames/`.

**Optional — rebuild the .blend from scripts** (only if the committed .blend
ever fails to load/render; takes ~15 s, overwrites `scenes/cables.blend`):
```
!cd /content/blender-cables-short && /content/blender-4.2.23-linux-x64/blender --background --python scripts/build_all.py
```
